# Importing Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

Separating provided columns for Set, Ratings, Amount, LGD and EAD from Borrowers Rating File

In [ ]:
def load_borrower_ratings(path="borrower_ratings.xlsx"):
    # Many original notebooks used header=1; keep same to align with data layout
    raw = pd.read_excel(path, header=1)
    borrower_col = raw.columns[0]
    other_cols = list(raw.columns[1:])

    # Original file split into 5 blocks of 10 columns each
    num_cols = other_cols[0:10]
    alpha_cols = other_cols[10:20]
    amount_cols = other_cols[20:30]
    lgd_cols = other_cols[30:40]
    ead_cols = other_cols[40:50]

    def make_block(df, cols):
        out = df[[borrower_col] + cols].copy()
        out = out.rename(columns={borrower_col: "Borrower_Number"})
        new_cols = ["Borrower_Number"] + [f"Year{i}" for i in range(1, 11)]
        out.columns = new_cols
        return out

    df_numeric = make_block(raw, num_cols)
    df_alpha   = make_block(raw, alpha_cols)
    df_amount  = make_block(raw, amount_cols)
    df_lgd     = make_block(raw, lgd_cols)
    df_ead     = make_block(raw, ead_cols)

    # Also create a cleaned copy of raw with Year1..Year10 mapping for any "Year 1" or "Year_1" naming
    df_clean = raw.copy()
    for i in range(1, 11):
        for col in (f"Year {i}", f"Year_{i}", f"Year{i}"):
            if col in df_clean.columns:
                df_clean = df_clean.rename(columns={col: f"Year{i}"})
                break

    return df_clean, df_numeric, df_alpha, df_amount, df_lgd, df_ead

# Load borrower ratings
df, df_numeric, df_alpha, df_amount, df_lgd, df_ead = load_borrower_ratings("borrower_ratings.xlsx")
print(f"Total borrowers: {len(df_numeric)}")

#**Exploratory Data Analysis**

In [ ]:
RATING_CATEGORIES = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'C', 'D']
rating_to_num = {r: i+1 for i, r in enumerate(RATING_CATEGORIES)}
num_to_rating = {i+1: r for i, r in enumerate(RATING_CATEGORIES)}

**Additional Settings**

In [ ]:
for i in range(1, 11):
    col = f"Year{i}"
    if col in df.columns:
        df[col] = df[col].map(num_to_rating).fillna(df[col])

# Build rating counts per year
rating_counts = {}
for i in range(1, 11):
    col = f"Year{i}"
    counts = df[col].value_counts()
    rating_counts[f"Year{i}"] = {r: int(counts.get(r, 0)) for r in RATING_CATEGORIES}

rating_data = pd.DataFrame.from_dict(rating_counts, orient="index")[RATING_CATEGORIES]

**Graph 1 - Stacked Bar Chart: Ratings Over Time**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(RATING_CATEGORIES)))
bottom_vals = np.zeros(len(rating_data))

# For x positions use numpy arange to ensure proper numeric x positions for text placement
x_positions = np.arange(len(rating_data.index))

for idx, rating in enumerate(RATING_CATEGORIES):
    values = rating_data[rating].values
    bars = ax.bar(x_positions, values, bottom=bottom_vals, label=rating, color=colors[idx])

    # Add labels inside each stacked segment (only when segment height > 0)
    for bar_i, bar in enumerate(bars):
        height = bar.get_height()
        if height > 0:
            y = bar.get_y() + height / 2.0
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                y,
                str(int(height)),
                ha='center', va='center', fontsize=8, color='black'
            )
    bottom_vals += values  # update bottom for next stack

# Set x-ticks to year labels
ax.set_xticks(x_positions)
ax.set_xticklabels([f"Year{i}" for i in range(1, 11)], rotation=0)
ax.set_ylabel("Number of Borrowers")
ax.set_xlabel("Year")
ax.set_title("Borrower Rating Distribution Over Years")
ax.legend(title="Rating", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

**Graph 2 - Bar charts for rating distributions**

In [ ]:
order = RATING_CATEGORIES
y1 = df["Year1"].value_counts().reindex(order, fill_value=0)
y10 = df["Year10"].value_counts().reindex(order, fill_value=0)
df_viz = pd.DataFrame({"Year1": y1, "Year10": y10}).reset_index().melt(id_vars="index", var_name="Year", value_name="Borrowers")
df_viz.columns = ["Rating", "Year", "Borrowers"]

plt.style.use("seaborn-v0_8-whitegrid")
plt.figure(figsize=(12, 6))
ax2 = sns.barplot(data=df_viz, x="Rating", y="Borrowers", hue="Year", edgecolor="black", linewidth=0.5)
plt.title("Portfolio Migration: Rating Distribution (Year1 vs Year10)", fontsize=14, fontweight="bold")
plt.xlabel("Credit Rating", fontsize=12)
plt.ylabel("Number of Borrowers", fontsize=12)
plt.legend(title="Timeline", title_fontsize=11, loc="upper right")
for container in ax2.containers:
    ax2.bar_label(container, padding=3, fontsize=9, fontweight="bold")
insight_text = (f"Key Insight:\\nDefaulters ('D') increased from {y1['D']} to {y10['D']}.\\n"
                f"High Grade ('AAA') dropped from {y1['AAA']} to {y10['AAA']}.")
plt.xticks(rotation=0)
plt.text(0.02, 0.95, insight_text, transform=ax2.transAxes, fontsize=10, bbox=dict(facecolor="white", alpha=0.9, edgecolor="gray"))
plt.tight_layout()
plt.show()

**Graph 3 - Heatmap for Rating Transition Matrix**

In [ ]:
for i in range(1, 11):
    col = f"Year{i}"
    if col in df_alpha.columns:
        df_alpha[col] = df_alpha[col].map(num_to_rating).fillna(df_alpha[col])

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()
for i in range(1, 10):
    from_col = f"Year{i}"
    to_col = f"Year{i+1}"
    transition = pd.crosstab(df_alpha[from_col], df_alpha[to_col], normalize="index").reindex(index=RATING_CATEGORIES, columns=RATING_CATEGORIES, fill_value=0)
    sns.heatmap(transition, annot=True, cmap="Blues", fmt=".2f", ax=axes[i-1], cbar=False, linewidths=0.5, linecolor="gray")
    axes[i-1].set_title(f"Transition: Year{i} to Year{i+1}")
    axes[i-1].set_xlabel(f"Year{i+1} Rating")
    axes[i-1].set_ylabel(f"Year{i} Rating")
plt.tight_layout()
plt.show()

**Graph 4 - Boxplot for LGD by ratings class**

In [ ]:
palette = sns.color_palette("Set2", n_colors=8)
# Create a mapping rating -> color
rating_color_map = {r: palette[i] for i, r in enumerate(RATING_CATEGORIES)}

fig, axes = plt.subplots(2, 5, figsize=(25, 10), sharey=True)
axes = axes.flatten()

for i in range(1, 11):
    ax = axes[i-1]
    year_col = f"Year{i}"
    # For boxplot, we want each rating box to have its own color.
    # We'll pass a palette mapping to seaborn.boxplot using a dict.
    sns.boxplot(x=df_alpha[year_col], y=df_lgd[year_col], order=RATING_CATEGORIES, palette=rating_color_map, ax=ax)
    ax.set_title(f"LGD Distribution by Rating (Year {i})")
    ax.set_ylabel("LGD (%)" if i == 1 else "")  # only first subplot shows ylabel to reduce clutter
    ax.set_xlabel("Rating")
    # Improve layout: rotate xticks slightly for readability
    ax.set_xticklabels(RATING_CATEGORIES, rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

#**Creating Transition Matrices**

In [ ]:
def compute_transition_matrix_matrix(from_year, to_year, df_ratings):
    from_col = f"Year{from_year}"
    to_col = f"Year{to_year}"
    counts = pd.crosstab(df_ratings[from_col], df_ratings[to_col], normalize="index")
    counts = counts.reindex(index=RATING_CATEGORIES, columns=RATING_CATEGORIES, fill_value=0)
    return counts

transition_matrices = {}
for y in range(2, 11):
    tm = compute_transition_matrix_matrix(y-1, y, df_alpha)
    transition_matrices[y] = tm
    print("\n" + "*"*80 + "\n")
    print(f"TRANSITION MATRIX: YEAR {y-1} → YEAR {y}")
    print("\n" + "*"*80)
    print(tm.round(4))

#**Calculating Probabilities of Default**

In [ ]:
def calc_pd_for_year(year, df_ratings, transition_matrix):
    prob_defaults_by_rating = transition_matrix['D'].to_dict()
    dist = df_ratings[f"Year{year}"].value_counts(normalize=True)
    portfolio_pd = 0.0
    for rating in RATING_CATEGORIES:
        if rating in dist.index and rating in prob_defaults_by_rating:
            portfolio_pd += dist[rating] * prob_defaults_by_rating[rating]
    return prob_defaults_by_rating, portfolio_pd

prob_default_results = {}
print("\n{:<6} {:<8} {:<8} {:<8} {:<8} {:<8} {:<8} {:<8} {:<8} {:<12}".format(
    "Year", "AAA", "AA", "A", "BBB", "BB", "B", "C", "D", "Portfolio PD"))
print("-" * 100)
for year in range(2, 11):
    pd_by_rating, portfolio_pd = calc_pd_for_year(year, df_alpha, transition_matrices[year])
    prob_default_results[year] = {'by_rating': pd_by_rating, 'portfolio': portfolio_pd}
    print("{:<6} {:<8.4f} {:<8.4f} {:<8.4f} {:<8.4f} {:<8.4f} {:<8.4f} {:<8.4f} {:<8.4f} {:<12.4f}".format(
        year,
        pd_by_rating['AAA'],
        pd_by_rating['AA'],
        pd_by_rating['A'],
        pd_by_rating['BBB'],
        pd_by_rating['BB'],
        pd_by_rating['B'],
        pd_by_rating['C'],
        pd_by_rating['D'],
        portfolio_pd
    ))

#**Calculating Expected Losses**

In [ ]:
def calc_expected_loss_for_year(year, df_ratings, df_lgd_data, df_ead_data, pd_by_rating):
    col = f"Year{year}"
    el_by_rating = {}
    portfolio_el = 0.0
    for rating in RATING_CATEGORIES:
        mask = (df_ratings[col] == rating)
        if mask.sum() == 0:
            el_by_rating[rating] = 0.0
            continue
        if rating == "D":
            prob_default = 1.0
        else:
            prob_default = pd_by_rating.get(rating, 0.0)
        avg_lgd = df_lgd_data[col][mask].mean() / 100.0
        total_ead = df_ead_data[col][mask].sum()
        el_rating = prob_default * avg_lgd * total_ead
        el_by_rating[rating] = el_rating
        portfolio_el += el_rating
    return el_by_rating, portfolio_el

el_results = {}
print("\n{:<6} {:>12} {:>12} {:>12} {:>12} {:>12} {:>12} {:>12} {:>12} {:>15}".format(
    "Year", "AAA", "AA", "A", "BBB", "BB", "B", "C", "D", "Portfolio EL"))
print("-" * 150)
for year in range(2, 11):
    pd_for_year = prob_default_results[year]['by_rating']
    el_by_rating, portfolio_el = calc_expected_loss_for_year(year, df_alpha, df_lgd, df_ead, pd_for_year)
    el_results[year] = {'by_rating': el_by_rating, 'portfolio': portfolio_el}
    print("{:<6} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>12,.4f} {:>15,.4f}".format(
        year,
        el_by_rating["AAA"],
        el_by_rating["AA"],
        el_by_rating["A"],
        el_by_rating["BBB"],
        el_by_rating["BB"],
        el_by_rating["B"],
        el_by_rating["C"],
        el_by_rating["D"],
        portfolio_el
    ))

#**Calculating Cumulative Probabilities of Default**

In [ ]:
def calc_cumulative_pd(end_year, df_ratings, transition_matrices_local):
    year1_dist = df_ratings['Year1'].value_counts(normalize=True)
    survival_probs = {r: 1.0 for r in RATING_CATEGORIES}
    for y in range(2, end_year+1):
        tm = transition_matrices_local[y]
        for rating in RATING_CATEGORIES:
            if rating in tm.index:
                survival_probs[rating] *= (1 - tm.loc[rating, 'D'])
    portfolio_cum_pd = 0.0
    for rating in RATING_CATEGORIES:
        if rating in year1_dist.index:
            weight = year1_dist[rating]
            cum_pd_rating = 1 - survival_probs[rating]
            portfolio_cum_pd += weight * cum_pd_rating
    return survival_probs, portfolio_cum_pd

surv5, cum_pd5 = calc_cumulative_pd(5, df_alpha, transition_matrices)
surv10, cum_pd10 = calc_cumulative_pd(10, df_alpha, transition_matrices)

print("\nCUMULATIVE PD BY RATING (Years 1-5):")
for rating in RATING_CATEGORIES:
    print(f"{rating:<6}: {(1 - surv5[rating])*100:>6.2f}%")
print(f"\nPORTFOLIO CUMULATIVE PD (Years 1-5): {cum_pd5*100:.4f}%")

print("\nCUMULATIVE PD BY RATING (Years 1-10):")
for rating in RATING_CATEGORIES:
    print(f"{rating:<6}: {(1 - surv10[rating])*100:>6.2f}%")
print(f"\nPORTFOLIO CUMULATIVE PD (Years 1-10): {cum_pd10*100:.4f}%")

**Values for Stress Testing Scenarios**

In [ ]:
stress = pd.read_csv("Stress_Test_Scenarios.csv", header=2)
group11 = stress[stress["Group No"] == 11].copy().drop(columns=["Group No"]).reset_index(drop=True)

# Scenario 1: LGD multipliers (Year1..Year10)
scenario1_cols_space = [f"Year {i}" for i in range(1, 11)]
scenario1_cols_no_space = [f"Year{i}" for i in range(1, 11)]
if all(c in group11.columns for c in scenario1_cols_space):
    lgd_multipliers_row = group11[scenario1_cols_space].iloc[0].values
else:
    lgd_multipliers_row = group11[scenario1_cols_no_space].iloc[0].values

# Scenario 2: rating migrations mapping Year9->Year10
rating_map_cols = ["AAA", "AA", "A", "BBB", "BB", "B", "C", "D"]
df_scenario2 = group11[rating_map_cols].copy()
scenario2_map = df_scenario2.iloc[0].to_dict()

print("\nStress Scenarios loaded for Group 11")
print("\nScenario 1 - LGD Multipliers:")
print(lgd_multipliers_row)
print("\nScenario 2 - Rating Migrations (Year9 -> Year10):")
print(scenario2_map)

#**Stress Testing - LGD Stress**

In [ ]:
print("\nLGD Stress Multipliers:")
for i, mult in enumerate(lgd_multipliers_row, 1):
    print(f"  Year {i}: {mult}x")

def calc_stressed_expected_loss(year, df_ratings, df_lgd_data, df_ead_data, pd_by_rating, lgd_multiplier):
    col = f"Year{year}"
    el_by_rating = {}
    portfolio_el = 0.0
    for rating in RATING_CATEGORIES:
        mask = (df_ratings[col] == rating)
        if mask.sum() > 0:
            prob_default = pd_by_rating.get(rating, 0.0)
            stressed_lgd_series = np.minimum(df_lgd_data[col][mask] * lgd_multiplier, 100)
            stressed_lgd_avg = stressed_lgd_series.mean() / 100.0
            avg_ead = df_ead_data[col][mask].mean()
            el_rating = prob_default * stressed_lgd_avg * avg_ead
            el_by_rating[rating] = el_rating
            total_ead = df_ead_data[col][mask].sum()
            stressed_lgd_portfolio = stressed_lgd_avg
            portfolio_el += prob_default * stressed_lgd_portfolio * total_ead
        else:
            el_by_rating[rating] = 0.0
    return el_by_rating, portfolio_el

stressed_el_results = {}
print("\n{:<6} {:<20} {:<20} {:<20} {:<12}".format("Year", "Normal EL (₹ Cr)", "Stressed EL (₹ Cr)", "Difference (₹ Cr)", "% Change"))
print("-" * 110)
years = list(range(2, 11))
for year in years:
    lgd_mult = lgd_multipliers_row[year - 1]
    _, stressed_port_el = calc_stressed_expected_loss(year, df_alpha, df_lgd, df_ead, prob_default_results[year]['by_rating'], lgd_mult)
    normal_el = el_results[year]['portfolio']
    difference = stressed_port_el - normal_el
    pct_change = (difference / normal_el * 100) if normal_el > 0 else 0.0
    stressed_el_results[year] = stressed_port_el
    print("{:<6} {:<20.4f} {:<20.4f} {:<20.4f} {:>10.2f}%".format(year, normal_el, stressed_port_el, difference, pct_change))

normal_els = [el_results[y]['portfolio'] for y in years]
stressed_els = [stressed_el_results[y] for y in years]
pct_increases = [(stressed_els[i] - normal_els[i]) / normal_els[i] * 100 if normal_els[i] != 0 else 0.0 for i in range(len(years))]

print("\nKEY OBSERVATIONS - LGD STRESS SCENARIO:")
avg_increase = np.mean(pct_increases)
max_increase = np.max(pct_increases)
max_year = years[np.argmax(pct_increases)]
print(f"  - Average EL increase: {avg_increase:.2f}%")
print(f"  - Maximum EL increase: {max_increase:.2f}% (Year {max_year})")
print(f"  - Total stressed EL (Year 10): ₹ {stressed_els[-1]:,.4f} Crore")

#**Stress Testing - Slippage of Returns**

In [ ]:
print("\nRating Migration (Year9 → Year10 under stress):")
for k, v in scenario2_map.items():
    print(f"  {k} → {v}")

def calc_migration_stress_el(df_ratings, df_lgd_data, df_ead_data, migration_map):
    year9 = df_ratings['Year9'].copy()
    stressed_year10 = year9.map(migration_map)
    total_el = 0.0
    el_by_original = {r: 0.0 for r in RATING_CATEGORIES}
    pd_matrix_year10 = transition_matrices[10]
    for idx in df_ratings.index:
        orig = year9.iloc[idx]
        stressed = stressed_year10.iloc[idx]
        prob_default = pd_matrix_year10.loc[stressed, 'D'] if stressed in pd_matrix_year10.index else 0.0
        lgd_val = df_lgd_data['Year10'].iloc[idx] / 100.0
        ead_val = df_ead_data['Year10'].iloc[idx]
        borrower_el = prob_default * lgd_val * ead_val
        total_el += borrower_el
        el_by_original[orig] += borrower_el
    return total_el, el_by_original, stressed_year10

migration_stressed_el, el_by_orig, stressed_year10_ratings = calc_migration_stress_el(df_alpha, df_lgd, df_ead, scenario2_map)
normal_year10_el = el_results[10]['portfolio']
difference = migration_stressed_el - normal_year10_el
pct_change = (difference / normal_year10_el * 100) if normal_year10_el > 0 else 0.0

print("\n" + "="*80)
print("RESULTS - RATING MIGRATION STRESS:")
print("="*80)
print(f"Normal Year 10 EL (₹ Crore):     {normal_year10_el:,.4f}")
print(f"Stressed Year 10 EL (₹ Crore):   {migration_stressed_el:,.4f}")
print(f"Difference (₹ Crore):            {difference:,.4f}")
print(f"Percentage Change:               {pct_change:.2f}%")

print("\n" + "-"*80)
print("RATING DISTRIBUTION COMPARISON (Year 10):")
print("-"*80)
print(f"{'Rating':<10} {'Normal Count':<15} {'Stressed Count':<15} {'Change':<10}")
normal_dist_y10 = df_alpha['Year10'].value_counts().reindex(RATING_CATEGORIES, fill_value=0)
stressed_dist_y10 = stressed_year10_ratings.value_counts().reindex(RATING_CATEGORIES, fill_value=0)
for r in RATING_CATEGORIES:
    ncnt = normal_dist_y10[r]
    scnt = stressed_dist_y10[r]
    chg = scnt - ncnt
    print(f"{r:<10} {ncnt:<15} {scnt:<15} {chg:+d}")

print("\nKEY OBSERVATIONS - RATING MIGRATION STRESS:")
downgrades = upgrades = no_change = 0
for idx in df_alpha.index:
    orig_idx = RATING_CATEGORIES.index(df_alpha['Year9'].iloc[idx])
    stressed_idx = RATING_CATEGORIES.index(stressed_year10_ratings.iloc[idx])
    if stressed_idx > orig_idx:
        downgrades += 1
    elif stressed_idx < orig_idx:
        upgrades += 1
    else:
        no_change += 1

print(f"  - Downgrades: {downgrades} borrowers ({downgrades/len(df_alpha)*100:.1f}%)")
print(f"  - Upgrades: {upgrades} borrowers ({upgrades/len(df_alpha)*100:.1f}%)")
print(f"  - No change: {no_change} borrowers ({no_change/len(df_alpha)*100:.1f}%)")
print(f"  - Net increase in defaults: {stressed_dist_y10['D'] - normal_dist_y10['D']}")

#**Stability Analysis across the years (Retention Rate)**

In [ ]:
stability = {r: [] for r in RATING_CATEGORIES}
for y in range(1, 10):
    col_curr = f"Year{y}"
    col_next = f"Year{y+1}"
    tm = pd.crosstab(df[col_curr], df[col_next], normalize="index")
    for r in RATING_CATEGORIES:
        if r in tm.index and r in tm.columns:
            stability[r].append(tm.loc[r, r])
        else:
            stability[r].append(0.0)
avg_stability = {r: np.mean(vals) for r, vals in stability.items()}
stability_df = pd.DataFrame(list(avg_stability.items()), columns=["Rating", "Avg_Retention_Rate"]).sort_values(by="Avg_Retention_Rate", ascending=False)
most_stable = stability_df.iloc[0]['Rating']
highest_score = stability_df.iloc[0]['Avg_Retention_Rate']
print("\nStability Rankings (Average Retention Rate):")
print(f"\n>>> CONCLUSION: The most stable rating category is '{most_stable}' with an average retention rate of {highest_score:.2%}.")

plt.figure(figsize=(10, 6))
plt.style.use("seaborn-v0_8-whitegrid")
colors = ['#2ecc71' if x == most_stable else '#3498db' for x in stability_df['Rating']]
ax = sns.barplot(x="Rating", y="Avg_Retention_Rate", data=stability_df, palette=colors, edgecolor="black")
plt.title("Stability Analysis: Average Annual Retention Rate (10-Year Period)", fontsize=14, fontweight="bold")
plt.ylabel("Retention Rate (Probability of Staying)", fontsize=12)
plt.xlabel("Credit Rating", fontsize=12)
plt.ylim(0, 1.1)
for i, v in enumerate(stability_df['Avg_Retention_Rate']):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

#**Significant Risk Shifters (Downgrades/Upgrades & WARF)**

In [ ]:
rs_results = []
for y in range(1, 10):
    curr = df[f"Year{y}"].map(rating_to_num)
    nxt = df[f"Year{y+1}"].map(rating_to_num)
    downgrades = (nxt > curr).sum()
    upgrades = (nxt < curr).sum()
    total = len(df)
    net_downgrade_intensity = (downgrades - upgrades) / total
    warf_curr = curr.mean()
    warf_next = nxt.mean()
    warf_shift = warf_next - warf_curr
    rs_results.append({
        "Period": f"Y{y}->Y{y+1}",
        "Downgrades": downgrades,
        "Upgrades": upgrades,
        "Net_Downgrade_intensity": net_downgrade_intensity,
        "WARF_Score_Start": warf_curr,
        "WARF_Shift": warf_shift
    })

rs_df = pd.DataFrame(rs_results)
max_idx = rs_df['Net_Downgrade_intensity'].idxmax()
worst_period = rs_df.loc[max_idx, 'Period']
worst_val = rs_df.loc[max_idx, 'Net_Downgrade_intensity']

print("\nRisk Shift Metrics (Year-over-Year):")
print(rs_df[['Period', 'Downgrades', 'Upgrades', 'Net_Downgrade_intensity', 'WARF_Shift']].round(4))
print(f"\n>>> IDENTIFIED SIGNIFICANT SHIFT: The most significant risk shift occurred during {worst_period}.")
print(f"    Reason: Net Downgrade Intensity peaked at {worst_val:.2%} (Highest excess of downgrades over upgrades).")

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(x='Period', y='Net_Downgrade_intensity', data=rs_df, ax=ax1, palette='viridis')
ax1.set_ylabel('Net Downgrade', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_title('Risk Shift Analysis: Net Downgrade and WARF Shift Over Time')
ax2 = ax1.twinx()
sns.lineplot(x='Period', y='WARF_Shift', data=rs_df, ax=ax2, color='red', marker='o', linewidth=2)
ax2.set_ylabel('WARF Shift', color='red')
ax2.tick_params(axis='y', labelcolor='red')
plt.tight_layout()
plt.show()